# Install Dependencies

In [1]:
# install standard libraries
#%pip install tensorflow

In [2]:
#%pip install opencv-python

# Importing libraries

In [3]:
#import standard libraries
import cv2, random , os , numpy as np , matplotlib.pyplot as plt

In [4]:
# import tensorflow dependencies - functional api 
from tensorflow.keras.models import Model # Model(inp = [inp image, verifi img],out=[0,1])
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten 
# layer- custom layers, conv2d - convolution , dense - fully connected layer, max pooling - pull out layers and shrink 
#  input - define what were passing to model , flatten- flows data from layers

import tensorflow as tf


# CREATE FOLDERS

In [5]:
# create 3 folders(paths) : anchor(input) / positive(verification image) / negative(verification image)
#setup paths 

pos_path = os.path.join('data','positive') # data/positive
neg_path = os.path.join('data','negative') #data/negative
anchor_path = os.path.join('data','anchor') #data/anchor


In [6]:
# make directories 
'''
os.makedirs(pos_path)
os.makedirs(neg_path)
os.makedirs(anchor_path)
'''

'\nos.makedirs(pos_path)\nos.makedirs(neg_path)\nos.makedirs(anchor_path)\n'

## Collect positive and Anchors

In [7]:
# import uuuid(universally unique identifier) to generate unique image names 
import uuid 

In [8]:
# uuid1, uuid2, uuid3, uuid4, uuid5  - formats you can use  
uuid.uuid1()

UUID('6536f093-40ab-11f1-8a17-40490f13fa1a')

In [9]:
'{}.jpg'.format(uuid.uuid1())

'65480817-40ab-11f1-9ed3-40490f13fa1a.jpg'

In [10]:
os.path.join(anchor_path,'{}.jpg'.format(uuid.uuid1())) 

'data\\anchor\\655f7708-40ab-11f1-88c0-40490f13fa1a.jpg'

In [11]:
# connect to webcam 
import cv2
cap = cv2.VideoCapture(0)
while cap.isOpened(): # loop through every frame in webcam 
    ret , frame = cap.read() # returns val and actual frame

    frame = frame[120:120+250,200:200+250,:] # cut down frame to 250px - 250px 

    # collect anchors 
    if cv2.waitKey(1) & 0XFF == ord('a'):
        # save frame to respective folders using uuid 
        imgname = os.path.join(anchor_path,'{}.jpg'.format(uuid.uuid1())) # unique faile path 
        cv2.imwrite(imgname,frame)


    # collect positives
    if cv2.waitKey(1) & 0XFF == ord('p'):
        # save frame to respective folders using uuid 
        imgname = os.path.join(pos_path,'{}.jpg'.format(uuid.uuid1())) # unique faile path 
        cv2.imwrite(imgname,frame)



    # show image back to screen    
    cv2.imshow('Image Collection', frame) 
    
    # press q to break webcam connection
    if cv2.waitKey(1) & 0XFF == ord('q'):
        break


#release the webcam 
cap.release()
# close the frame showing image 
cv2.destroyAllWindows()


In [12]:
# click a - anchor , p- positive 
#plt.imshow(frame)
frame.shape # 250px x 250px x 3 channels

(250, 250, 3)

In [13]:
# slicing to get right shape [yaxis,xaixs]
#plt.imshow(frame[120:120+250,200:200+250,:])

# LOAD AND PREPROCESS IMAGES 

In [14]:
# get image directories 
# 3 vars = grab all images in the specific folder 
anchor = tf.data.Dataset.list_files(anchor_path+'\*.jpg').take(90)# get every file with \xxxxxx.jpg only 90 images
positive = tf.data.Dataset.list_files(pos_path+'\*.jpg').take(90)# number of samples should be same it only loads path not actual images
negative = tf.data.Dataset.list_files(neg_path+'\*.jpg').take(90)

In [15]:
anchor_path+'\*.jpg'

'data\\anchor\\*.jpg'

In [16]:
dir_test = anchor.as_numpy_iterator()
dir_test.next() # get next file( image id )

b'data\\anchor\\d0e22978-381f-11f1-b688-40490f13fa1a.jpg'

## PREPROCESSING - SCALE AND RESIZE 
fxn to load image and resize it  and convert image val to 0-255 to 0-1

In [17]:
def preprocess(file_path):
    byte_img = tf.io.read_file(file_path)  # read img using file path treeated using bytes like obj 
    img = tf.io.decode_jpeg(byte_img)          # decode jpeg -> load img 
    
    img = tf.image.resize(img,(100,100))    # resize img -> preprocessing (100px, 100px, 3 channels) as per paper 
    img = img / 255.0              # divide it by 255 so it performs scaling and it returnns the image 0-1
    return img

In [18]:
img = preprocess('data\\anchor\\41833586-381f-11f1-80e5-40490f13fa1a.jpg')

In [19]:
img.shape

TensorShape([100, 100, 3])

In [20]:
img.numpy().min()

0.014705882

In [21]:
img.numpy().max()

0.89730394

In [22]:
#plt.imshow(img)

## CREATE OUTPUT FEATURE 

In [23]:
# use 
tf.ones_like(1)

<tf.Tensor: shape=(), dtype=int32, numpy=1>

In [24]:
tf.ones_like([1,1,3,4,555.5,666.0]) # - > set of ones not different vals 

<tf.Tensor: shape=(6,), dtype=float32, numpy=array([1., 1., 1., 1., 1., 1.], dtype=float32)>

In [25]:
# pass anchot + pos img as input // otp -> array os ones 
# same for negatives -> otp 0 

In [26]:
tf.ones(len(anchor))

<tf.Tensor: shape=(90,), dtype=float32, numpy=
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1.], dtype=float32)>

In [27]:
tf.zeros(len(anchor))

<tf.Tensor: shape=(90,), dtype=float32, numpy=
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.], dtype=float32)>

In [28]:
tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor))) # putting data into data loader 

<TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.float32, name=None)>

In [29]:
# anc +pos -> 1
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
# anc + neg -> 0
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [30]:
data

<ConcatenateDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [31]:
samples  = data.as_numpy_iterator()

In [32]:
samples.next() # keep iterating to dpoints

(b'data\\anchor\\43a67ca7-381f-11f1-8d04-40490f13fa1a.jpg',
 b'data\\positive\\53372d9b-3820-11f1-9bbf-40490f13fa1a.jpg',
 1.0)

## BUILD TRAIN AND TEST PARTITION

In [33]:
'''(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg', # input 
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',  # valid 
 np.float32(1.0))'''
def preprocess_twin(input_image, validation_image, label):
    return (preprocess(input_image), preprocess(validation_image), label)

In [34]:
res = preprocess_twin(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg',
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',
 np.float32(1.0))

In [35]:
res[0] # converts 100 , 100 , 3

<tf.Tensor: shape=(100, 100, 3), dtype=float32, numpy=
array([[[0.76985294, 0.7776961 , 0.75808823],
        [0.7590686 , 0.76691175, 0.7473039 ],
        [0.76593137, 0.782598  , 0.75710785],
        ...,
        [0.8622549 , 0.89362746, 0.8504902 ],
        [0.8784314 , 0.9098039 , 0.8607843 ],
        [0.8627451 , 0.8862745 , 0.83137256]],

       [[0.7683824 , 0.7713235 , 0.73995095],
        [0.7588235 , 0.76862746, 0.74019605],
        [0.76862746, 0.77843136, 0.75      ],
        ...,
        [0.8757353 , 0.90661764, 0.85784316],
        [0.8620098 , 0.8870098 , 0.84117645],
        [0.8610294 , 0.88529414, 0.82892156]],

       [[0.76715684, 0.76715684, 0.72205883],
        [0.7767157 , 0.7769608 , 0.73382354],
        [0.7772059 , 0.77818626, 0.7409314 ],
        ...,
        [0.8661765 , 0.8904412 , 0.8512255 ],
        [0.8691176 , 0.8926471 , 0.85343134],
        [0.8762255 , 0.89093137, 0.84632355]],

       ...,

       [[0.73995095, 0.7352941 , 0.7169118 ],
        [0.73

In [36]:
res[2]

1.0

In [37]:
#plt.imshow(res[0])

In [38]:
# build data loader pipeline 
data = data.map(preprocess_twin)
data = data.cache() # save processed data to ram 
data = data.shuffle(buffer_size = 1024) # pick 1024 dpoints 

In [39]:
sample = data.as_numpy_iterator()

In [40]:
eg = sample.next()

In [41]:
#plt.imshow(eg[0])

In [42]:
#plt.imshow(eg[1])

In [43]:
eg[2]

1.0

In [44]:
len(data)

180

In [45]:
round((70/100)*180)

126

In [46]:
# train  partition 
train_data = data.take(round(len(data)*.7))  # 70% training data 
train_data = train_data.batch(8,drop_remainder=True) # pass through data as batch of 16
train_data = train_data.prefetch(4) # starts preprocessing next 4 images 

In [47]:
ts = train_data.as_numpy_iterator()

In [48]:
len(ts.next()[0]) # 8 images 

8

In [49]:
data

<ShuffleDataset element_spec=(TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [50]:
round(len(data)*.3)

54

In [51]:
# testing partition 
test_data = data.skip(round(len(data)*.7)) # skip first 70 % 
test_data = test_data.take(round(len(data)*.3)) # take only last 30 %
test_data = test_data.batch(8,drop_remainder=True)
test_data = test_data.prefetch(4)

# Model Engineering 

### BUILDING EMBEDDING LAYER 

In [52]:
inp = Input(shape=(100,100,3), name="input_image") # using 3 because we have colored img in paper 105,105,1 
inp # none-> batch size 

<KerasTensor: shape=(None, 100, 100, 3) dtype=float32 (created by layer 'input_image')>

In [53]:
c1 = Conv2D(64,(10,10), activation = 'relu')(inp)

In [54]:
c1 # shape = 91, 91 , 64 channels  bec of 100, 100 , 3 

<KerasTensor: shape=(None, 91, 91, 64) dtype=float32 (created by layer 'conv2d')>

In [55]:
m1 = MaxPooling2D(64,(2,2), padding = 'same')(c1)
m1

<KerasTensor: shape=(None, 46, 46, 64) dtype=float32 (created by layer 'max_pooling2d')>

In [56]:
# second block 
c2 = Conv2D(128,(7,7), activation='relu')(m1)
m2 = MaxPooling2D(64,(2,2), padding = 'same')(c2)

In [57]:
c2 # check on the paper 

<KerasTensor: shape=(None, 40, 40, 128) dtype=float32 (created by layer 'conv2d_1')>

In [58]:
m2

<KerasTensor: shape=(None, 20, 20, 128) dtype=float32 (created by layer 'max_pooling2d_1')>

In [59]:
# third block 
c3 = Conv2D(128,(4,4), activation='relu')(m2)
m3 = MaxPooling2D(64,(2,2), padding = 'same')(c3)

In [60]:
print(c3)
print(m3)

KerasTensor(type_spec=TensorSpec(shape=(None, 17, 17, 128), dtype=tf.float32, name=None), name='conv2d_2/Relu:0', description="created by layer 'conv2d_2'")
KerasTensor(type_spec=TensorSpec(shape=(None, 9, 9, 128), dtype=tf.float32, name=None), name='max_pooling2d_2/MaxPool:0', description="created by layer 'max_pooling2d_2'")


In [61]:
c4 = Conv2D(256,(4,4), activation = 'relu')(m3)

In [62]:
c4

<KerasTensor: shape=(None, 6, 6, 256) dtype=float32 (created by layer 'conv2d_3')>

In [63]:
6*6*256

9216

In [64]:
f1 = Flatten()(c4)
f1

<KerasTensor: shape=(None, 9216) dtype=float32 (created by layer 'flatten')>

In [65]:
d1 = Dense(4096, activation = 'sigmoid')(f1)
d1

<KerasTensor: shape=(None, 4096) dtype=float32 (created by layer 'dense')>

In [66]:
mod = Model(inputs=[inp], outputs=[d1], name='embedding')
mod.summary()

Model: "embedding"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 100, 100, 3)]     0         
                                                                 
 conv2d (Conv2D)             (None, 91, 91, 64)        19264     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 46, 46, 64)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 40, 40, 128)       401536    
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 20, 20, 128)      0         
 2D)                                                             
                                                                 
 conv2d_2 (Conv2D)           (None, 17, 17, 128)       26

In [67]:
def make_embedding():
    # create input layer 
    inp = Input(shape=(100,100,3), name="input_image")

# first block     
    # convolution layer + relu activation : convlay 2 imp things (64 = no of filters , 10px by 10px = filter shape )
    c1 = Conv2D(64,(10,10), activation = 'relu')(inp)

    # max pooling layer 
    m1 = MaxPooling2D(64,(2,2), padding = 'same')(c1)

    # inpu -> con+relu -> maxpool => conv + relu -> maxpool => conv +relu -> maxpool => coonv +relu -> fully connected + sigmoid l1 siamese distance 

 # second block 
    c2 = Conv2D(128,(7,7), activation='relu')(m1)
    m2 = MaxPooling2D(64,(2,2), padding = 'same')(c2)
    
# third block 
    c3 = Conv2D(128,(4,4), activation='relu')(m2)
    m3 = MaxPooling2D(64,(2,2), padding = 'same')(c3)

# final embedding block 
    c4 = Conv2D(256,(4,4), activation = 'relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation = 'sigmoid')(f1)



    # return embedding model imported using tensorflow.keras.models import Model
    return Model(inputs=[inp], outputs=[d1], name='embedding')

In [68]:
# build model 
embedding_model = make_embedding()

In [69]:
embedding_model.summary()

Model: "embedding"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_image (InputLayer)    [(None, 100, 100, 3)]     0         
                                                                 
 conv2d_4 (Conv2D)           (None, 91, 91, 64)        19264     
                                                                 
 max_pooling2d_3 (MaxPooling  (None, 46, 46, 64)       0         
 2D)                                                             
                                                                 
 conv2d_5 (Conv2D)           (None, 40, 40, 128)       401536    
                                                                 
 max_pooling2d_4 (MaxPooling  (None, 20, 20, 128)      0         
 2D)                                                             
                                                                 
 conv2d_6 (Conv2D)           (None, 17, 17, 128)       26

### BUILD DISTANCE LAYER 

In [70]:
class L1Dist(Layer): # custom layer (Layer class is passes)
    def __init__(self, **kwargs):
        super().__init__(**kwargs) # inheritance 

    # core fxn 
    def call(self, inputs):
        # Indexing [0] handles the nested list structure
        input_embedding = inputs[0]
        validation_embedding = inputs[1]
        
        return tf.math.abs(input_embedding - validation_embedding) # calculates similarity 


In [71]:
l1 = L1Dist()
l1

In [72]:
# l1(anchor_embedding , validation_embedding) -> combine it into a dense layer (fully connected layer )

### MAKE A SINGLE COMBINED SIAMESE MODEL 

In [73]:
input_img  = Input(name='input_image', shape = (100,100,3))
validation_img = Input(name = 'validation_image', shape = (100,100,3))

In [74]:
inp_emb = embedding_model(input_img)
inp_emb # 200,200,3 image -> 4096 units (feature vector )

<KerasTensor: shape=(None, 4096) dtype=float32 (created by layer 'embedding')>

In [75]:
val_emb = embedding_model(validation_img)
val_emb

<KerasTensor: shape=(None, 4096) dtype=float32 (created by layer 'embedding')>

In [76]:
# take siamese layer give 2 inputs -> 4096 feature vector -> dense layer  
siamese_layer = L1Dist()

In [77]:
dis = siamese_layer([inp_emb, val_emb])

In [78]:
classifier = Dense(1,activation = 'sigmoid')(dis)

In [79]:
classifier

<KerasTensor: shape=(None, 1) dtype=float32 (created by layer 'dense_2')>

In [80]:
siamese_network = Model(inputs=[input_img, validation_img], outputs= classifier,  name= 'SiameseNetwork')
siamese_network.summary()

Model: "SiameseNetwork"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_image (InputLayer)       [(None, 100, 100, 3  0           []                               
                                )]                                                                
                                                                                                  
 validation_image (InputLayer)  [(None, 100, 100, 3  0           []                               
                                )]                                                                
                                                                                                  
 embedding (Functional)         (None, 4096)         38960448    ['input_image[0][0]',            
                                                                  'validation_image[0

In [81]:
def make_siamese_model():
    # input img in nwk 
    input_img  = Input(name='input_image', shape = (100,100,3))

    # validation img in nwk
    validation_img = Input(name = 'validation_image', shape =(100,100,3))

    # take inputs pass them through embedding model then combine with sieamese model 

    # combine siamese distance components 
    siamese_layer = L1Dist()
    siamese_layer._name = 'distance'
    distances = siamese_layer([embedding_model(input_img), embedding_model(validation_img)])

    # classification layer 
    classifier = Dense(1,activation = 'sigmoid')(distances) # pass 4096 units in -> output as 1 unit only by applying sigmoid 

    return Model(inputs=[input_img, validation_img], outputs= classifier,  name= 'SiameseNetwork')

    

In [82]:
siamese_model = make_siamese_model()

In [83]:
siamese_model.summary()

Model: "SiameseNetwork"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_image (InputLayer)       [(None, 100, 100, 3  0           []                               
                                )]                                                                
                                                                                                  
 validation_image (InputLayer)  [(None, 100, 100, 3  0           []                               
                                )]                                                                
                                                                                                  
 embedding (Functional)         (None, 4096)         38960448    ['input_image[0][0]',            
                                                                  'validation_image[0

# MODEL TRAINING 

### SETUP LOSS FUNCTION AND OPTIMISER 

In [84]:
binary_cross_loss= tf.losses.BinaryCrossentropy() 

In [85]:
# define optimiser - adam optimisier 
opt = tf.keras.optimizers.Adam(1e-4) # 0.0001 learning rate 

### ESTABLISH CHECKPOINT CALLBACKS

In [86]:
checkpoint_dir  = './training_checkpoints' # define checkpoint directory to save all checkpoints 
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt') # ckpt1, ckpt2 , ckpt3
checkpoint = tf.train.Checkpoint(opt = opt , siamese_model = siamese_model) # optimiser , siamese model 

###  BUILD TRAIN STEP 
this will be used to train one batch of data 


input (first batch)-> model (training step)-> make pred -> cal loss(gradients) -> apply backpropagartion -> best modek 

In [87]:
test_batch = train_data.as_numpy_iterator()

In [88]:
batch1 = test_batch.next()

In [89]:
len(batch1) # made up of 3 comp actual / verif / output

3

In [90]:
len(batch1[0])

8

In [91]:
X = batch1[:2]

In [92]:
np.array(X).shape # 2 components (anch + pos/neg), each batch has 8 samples image shape = 100 100 3 channels

(2, 8, 100, 100, 3)

In [93]:
y = batch1[2]

In [94]:
y

array([0., 1., 0., 0., 1., 0., 1., 0.], dtype=float32)

In [95]:
# use tf.function decorator : compiles a function into a callable tensorflow graph 
@tf.function
def train_step(batch):
    #calculate gradients for neural network 
    with tf.GradientTape() as tape: 

        # get anchor / pos / neg images 
        X=batch[:2] # 0 - anchor image , 1 - pos/neg image

        # get target variavle 
        y = batch[2]

        # forward pass
        ypred = siamese_model(X,training = True)  # activate all layers set training to true 

        # cal loss 
        loss = binary_cross_loss(y , ypred) # y actual - y hat / pred
    tf.print("Loss:", loss) 

    # cal gradients which weight has made error and how much using chain rule 
    grad = tape.gradient(loss, siamese_model.trainable_variables)

    # calculate updated weights and apply to siamese model 
    opt.apply_gradients(zip(grad,siamese_model.trainable_variables)) # trainable vars ->. all vars that contributes in nwk

    # return loss
    return loss
        

### BUILD TRAINING LOOP 

In [96]:
# Re-create the checkpoint object to match what you saved
checkpoint = tf.train.Checkpoint(opt=opt, siamese_model=siamese_model)

# Load the weights you saved manually
checkpoint.restore(tf.train.latest_checkpoint('./training_checkpoints'))

print("Weights loaded! Model is now at the end of Epoch 10.")



Weights loaded! Model is now at the end of Epoch 10.


In [97]:
def train_loop(train_data,EPOCHS):

    # loop through epoch 
     for epoch in range(3,EPOCHS+1):
         print('\n EPOCH {}/{}'.format(epoch, EPOCHS)) # 1/100 or 3 / 100
         progbar = tf.keras.utils.Progbar(len(train_data))
         
        # loop through each batch 
         for idx, batch in enumerate(train_data):
             # run the train step here
             loss_val = train_step(batch)
             progbar.update(idx + 1, values=[("loss", loss_val)])    
             
        # save checkpoint for every 10 epochs 
         if epoch %2 ==0:
             checkpoint.save(file_prefix = checkpoint_prefix)

### TRAIN THE MODEL 

In [98]:
'''
EPOCHS = 10
train_loop(train_data,EPOCHS) 
'''

'\nEPOCHS = 10\ntrain_loop(train_data,EPOCHS) \n'

# EVALUATE THE MODEL USING : precision and recall 

### IMPORT METRICS

In [99]:
# import metric calculation 
from tensorflow.keras.metrics import Precision, Recall

In [100]:
# get a batch of test data 
test_input, test_val , y_true = test_data.as_numpy_iterator().next()

In [101]:
# grab one batch of data - 8 dpoints  
test_var = test_data.as_numpy_iterator().next()

In [102]:
len(test_var)

3

In [103]:
len(test_var[0])

8

In [104]:
len(test_var[1])

8

In [105]:
len(test_var[2])

8

In [106]:
test_input # input images 

array([[[[0.77156866, 0.7676471 , 0.7497549 ],
         [0.782598  , 0.7884804 , 0.7639706 ],
         [0.7767157 , 0.7855392 , 0.7639706 ],
         ...,
         [0.7970588 , 0.81764704, 0.7656863 ],
         [0.8117647 , 0.8235294 , 0.7862745 ],
         [0.81960785, 0.8235294 , 0.8       ]],

        [[0.7767157 , 0.7767157 , 0.74534315],
         [0.7764706 , 0.7796569 , 0.74828434],
         [0.7676471 , 0.777451  , 0.747549  ],
         ...,
         [0.79509807, 0.8068628 , 0.7558824 ],
         [0.8068628 , 0.80833334, 0.7683824 ],
         [0.80588233, 0.80588233, 0.77254903]],

        [[0.7705882 , 0.7705882 , 0.7392157 ],
         [0.7735294 , 0.7745098 , 0.74313724],
         [0.7737745 , 0.78357846, 0.7536765 ],
         ...,
         [0.7992647 , 0.810049  , 0.7654412 ],
         [0.7916667 , 0.7987745 , 0.7563726 ],
         [0.7987745 , 0.80465686, 0.7625    ]],

        ...,

        [[0.66862744, 0.6764706 , 0.627451  ],
         [0.66764706, 0.6745098 , 0.6343137 ]

In [107]:
test_val # validation images =/-

array([[[[0.00000000e+00, 3.92156886e-03, 0.00000000e+00],
         [0.00000000e+00, 6.86274515e-03, 0.00000000e+00],
         [0.00000000e+00, 6.86274515e-03, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 3.92156886e-03, 0.00000000e+00],
         [0.00000000e+00, 6.12745108e-03, 0.00000000e+00],
         [0.00000000e+00, 3.92156886e-03, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

        [[0.00000000e+00, 9.80392215e-04, 0.00000000e+00],
         [0.00000000e+00, 3.92156886e-03, 0.00000000e+00],
         [0.00000000e+00, 1.71568629e-03, 0.00000000e+00],
         ...,
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
         [

In [108]:
y_true

array([0., 1., 1., 1., 0., 0., 1., 0.], dtype=float32)

### MAKE PREDICTIONS 

In [109]:
# make predictions by passing test_input and test_val data to the model and predict if it verifies the person or not 
preds = siamese_model.predict([test_input, test_val])
preds

1/1 [==============================] - 12s 12s/step


array([[0.4994257 ],
       [0.4998954 ],
       [0.5001983 ],
       [0.50018597],
       [0.5010013 ],
       [0.49935293],
       [0.4999075 ],
       [0.49963653]], dtype=float32)

#### POST PROCESSING THE RESULTS

In [110]:
# convert confidence matrix it into 1 or 0 using a threshold value = 0.5
y_pred = [1 if i > 0.5 else 0 for i in preds] # list comprehension 
y_pred

[0, 0, 1, 1, 1, 0, 0, 0]

In [111]:
'''
 anotehr way 
 
 result = []
 for i in preds:
     if i > 0.5:
         result.append(1)
    else:
        result.append(0)
     
'''

'\n anotehr way \n \n result = []\n for i in preds:\n     if i > 0.5:\n         result.append(1)\n    else:\n        result.append(0)\n     \n'

In [112]:
y_true

array([0., 1., 1., 1., 0., 0., 1., 0.], dtype=float32)

In [113]:
y_pred

[0, 0, 1, 1, 1, 0, 0, 0]

### RECALL

In [114]:
# use metrics to check model performance 
# create metric object for recall
m = Recall()

# update states : calculating the recall value 
m.update_state(y_true, y_pred)

# return recall result 
m.result().numpy()

0.5

### PRECISION

In [115]:
# check for precision 
pcs = Precision()
pcs.update_state(y_true, y_pred)
pcs.result().numpy()

0.6666667

<B> HIGH RECALL + HIGH PRECISION = GOOD <B>

- model has been tested on a single batch only 
- cal precision and recall for the entire test dataset

### VISUALISE RESULTS 

In [116]:
'''
#plt.figure(figsize= (10,8))
# plotting images side by side (rows,columns,index of matrix)
plt.subplot(1,2,1)
plt.imshow(test_input[5]) # idex shoud be same for both inp and val 

#second subplot
plt.subplot(1,2,2)
plt.imshow(test_val[5])
plt.show()
'''

'\n#plt.figure(figsize= (10,8))\n# plotting images side by side (rows,columns,index of matrix)\nplt.subplot(1,2,1)\nplt.imshow(test_input[5]) # idex shoud be same for both inp and val \n\n#second subplot\nplt.subplot(1,2,2)\nplt.imshow(test_val[5])\nplt.show()\n'

# SAVE MODEL

In [117]:
# save model wights 
#siamese_model.save('siameseModel.h5')

In [118]:
# reload model
model = tf.keras.models.load_model('siameseModel.h5', custom_objects={'L1Dist': L1Dist, 'BinaryCrossentropy':tf.losses.BinaryCrossentropy})

In [119]:
#tf.keras.models.load_model??  Loads a model saved via `model.save()

In [120]:
# use reloded model to make predictions
model.predict([test_input, test_val])

1/1 [==============================] - 17s 17s/step


array([[0.49928743],
       [0.5000808 ],
       [0.5004417 ],
       [0.49938637],
       [0.5006864 ],
       [0.50036126],
       [0.4993888 ],
       [0.5019244 ]], dtype=float32)

In [121]:
model.summary()

Model: "SiameseNetwork"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_image (InputLayer)       [(None, 100, 100, 3  0           []                               
                                )]                                                                
                                                                                                  
 validation_image (InputLayer)  [(None, 100, 100, 3  0           []                               
                                )]                                                                
                                                                                                  
 embedding (Functional)         (None, 4096)         38960448    ['input_image[0][0]',            
                                                                  'validation_image[0

# REAL TIME TEST

### VERIFICATION FUNCTION

In [122]:
os.path.join('application_data','verification_images')

'application_data\\verification_images'

In [123]:
os.listdir(os.path.join('application_data','verification_images'))

['0098c63b-3820-11f1-8a76-40490f13fa1a.jpg',
 '075f4320-3820-11f1-a603-40490f13fa1a.jpg',
 '0aa99de3-3820-11f1-a0d9-40490f13fa1a.jpg',
 '0c63a983-3820-11f1-8e85-40490f13fa1a.jpg',
 '0ef6cb81-3820-11f1-ae3b-40490f13fa1a.jpg',
 '0f4cc538-3820-11f1-b22b-40490f13fa1a.jpg',
 '0f5b3657-3820-11f1-bc51-40490f13fa1a.jpg',
 '14c7c850-3820-11f1-b65e-40490f13fa1a.jpg',
 '163b575d-3820-11f1-a4e1-40490f13fa1a.jpg',
 '170d006a-3820-11f1-99f6-40490f13fa1a.jpg',
 '1a298bae-3820-11f1-a217-40490f13fa1a.jpg',
 '1ba2925b-3820-11f1-8027-40490f13fa1a.jpg',
 '1d124c83-3820-11f1-b82f-40490f13fa1a.jpg',
 '1fe963cc-3820-11f1-b0f0-40490f13fa1a.jpg',
 '24d5814e-3820-11f1-a307-40490f13fa1a.jpg',
 '25a3d6dd-3820-11f1-a972-40490f13fa1a.jpg',
 '2ca48b3e-3820-11f1-af51-40490f13fa1a.jpg',
 '323d3195-3820-11f1-b018-40490f13fa1a.jpg',
 '52c050a6-3820-11f1-afc4-40490f13fa1a.jpg',
 '535b9ad6-3820-11f1-8446-40490f13fa1a.jpg',
 '642ec463-3820-11f1-8a98-40490f13fa1a.jpg',
 '643bbfa6-3820-11f1-8b95-40490f13fa1a.jpg',
 '645c21e8

In [124]:
# 4 args :  siameseNN , metric above which pred is considered +ve class, ratio of +ve pred over total +ve samples(30/50)
def verify(model, detection_threshold, verification_threshold):
    # build results array: store results in array
    results = []
    # loop through every 50 images in verif folder (full cycle loop)
    for image in os.listdir(os.path.join('application_data','verification_images')):
        input_img = preprocess(os.path.join('application_data','input_image', 'input_image.jpg')) # loads image -> resive -> convert pcx to no
        validation_img = preprocess(os.path.join('application_data','verification_images', image)) # same for valid image

        # make pred 
        result = model.predict(list(np.expand_dims([input_img, validation_img], axis=1)))
        results.append(result)

    detection =  np.sum(np.array(results)>detection_threshold)
    
    verification = detection / len(os.listdir(os.path.join('application_data','verification_images')))
    verified = verified > verification_threshold 

    return results , verified
    
    

### OpenCV REAL TIME VERIFICATION

In [125]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret , frame = cap.read()
    
    frame = frame[120:120+250,200:200+250,:] # cut down frame to 250px - 250px

    cv2.imshow('Verification', frame) # frame name top bar of window 


    # trigger verification
    if cv2.waitKey(10) & 0xFF == ord('v'):
        # save inp image to application data/inp_image folder
        cv2.imwrite(os.path.join('application_data','input_image','input_image.jpg'), frame)

        # run verification 
        results , verified = verify( model,0.5,0.5) # threshold = 50%
        print(verified)
            
    

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 2s 2s/step


UnboundLocalError: local variable 'verified' referenced before assignment

In [ ]:
ord('q')